# LeetCode 3370 — Smallest Number With All Set Bits bằng Python

Link đề: https://leetcode.com/problems/smallest-number-with-all-set-bits/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation (thao tác bit), Math (toán)

Skeleton LeetCode:

```python
class Solution:
    def smallestNumber(self, n: int) -> int:
        ...
```

Ghi chú: ba ví dụ chính lấy từ đề; mọi đáp án trong notebook được kiểm chứng bằng code (kể cả vét cạn **toàn bộ** `n = 1..1000` và đối chiếu năm cách làm với nhau). Ví dụ tự dựng thêm được ghi rõ "tự dựng" — nếu cần khớp từng chữ thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho số nguyên dương `n`, tìm số `x` **nhỏ nhất** với `x >= n` sao cho biểu diễn **nhị phân** của `x` **chỉ gồm các bit `1`** (ví dụ `1`, `11`, `111`, ...).
2. Nắm các công cụ Python: `bin(x)`, phép dịch bit `<<` và `>>`, phép OR `|`, phương thức `int.bit_length()`, và mẹo kiểm tra "toàn bit 1" bằng `x & (x + 1) == 0`.
3. Đi từ cách **thử từng số** `n, n+1, ...` → **nhận ra các số toàn bit 1 là `2^p - 1`** → **vòng lặp dịch bit** → **`bit_length`** (một dòng) → **"làm đầy" bit bằng OR** → **tra bảng 10 số**.
4. Hiểu mấu chốt: các số toàn bit 1 chỉ là dãy `1, 3, 7, 15, 31, ...` (`2^p - 1`); đáp án là phần tử **đầu tiên** của dãy đó mà `>= n`. Nếu `n` có `b` bit thì đáp án là `2^b - 1`.
5. Tránh các lỗi kinh điển: kiểm tra `'0' not in bin(x)` (chuỗi `'0b'` luôn chứa `'0'`), lệch một đơn vị khi dùng `<=` thay `<`, sai thứ tự ưu tiên toán tử với `1 << n.bit_length() - 1`, và nhầm `2^p` với `2^p - 1`.

## 2. Đề bài & ví dụ

Cho một số nguyên **dương** `n`. Trả về số nguyên `x` **nhỏ nhất** thoả mãn:

- `x >= n`, và
- biểu diễn nhị phân của `x` **chỉ chứa bit đặt (set bit)**, nghĩa là mọi bit đều bằng `1`.

**Ví dụ 1**

- Input: `n = 5`
- Output: `7`
- Giải thích: `7` viết nhị phân là `"111"`. (Còn `5 = "101"`, `6 = "110"` đều có bit `0`.)

**Ví dụ 2**

- Input: `n = 10`
- Output: `15`
- Giải thích: `15` viết nhị phân là `"1111"`.

**Ví dụ 3**

- Input: `n = 3`
- Output: `3`
- Giải thích: `3` viết nhị phân là `"11"` — chính `n` đã toàn bit `1`, nên đáp án là `n`.

**Ràng buộc**

- `1 <= n <= 1000`

In [1]:
examples = [
    (5, 7),
    (10, 15),
    (3, 3),
]

def smallest_quick(n: int) -> int:
    """Bản nhanh để chạy thử ví dụ: thử từng số từ n trở lên."""
    x = n
    while bin(x)[2:].count("0") > 0:   # còn bit 0 thì thử số kế tiếp
        x += 1
    return x

for n, expected in examples:
    print(f"n={n} (nhị phân {bin(n)[2:]}) -> {smallest_quick(n)} (kỳ vọng {expected})")
    assert smallest_quick(n) == expected

n=5 (nhị phân 101) -> 7 (kỳ vọng 7)
n=10 (nhị phân 1010) -> 15 (kỳ vọng 15)
n=3 (nhị phân 11) -> 3 (kỳ vọng 3)


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân: `bin(x)` và "set bit"

Máy tính lưu số bằng các **bit** `0`/`1`. Một bit bằng `1` được gọi là **set bit** (bit đã đặt). `bin(x)` trả về chuỗi nhị phân **bắt đầu bằng `0b`**, ví dụ `bin(5) = '0b101'`. Muốn bỏ tiền tố, cắt lát `bin(5)[2:]` hoặc dùng `format(5, "b")`.

In [2]:
for x in (1, 5, 7, 10, 15):
    print(f"x={x:>2}: bin={bin(x):>8}  bỏ '0b' -> {bin(x)[2:]:>4}  format -> {format(x, 'b')}")

x= 1: bin=     0b1  bỏ '0b' ->    1  format -> 1
x= 5: bin=   0b101  bỏ '0b' ->  101  format -> 101
x= 7: bin=   0b111  bỏ '0b' ->  111  format -> 111
x=10: bin=  0b1010  bỏ '0b' -> 1010  format -> 1010
x=15: bin=  0b1111  bỏ '0b' -> 1111  format -> 1111


### 3.2. Dịch bit `<<` và `>>`

- `x << k` dịch các bit của `x` sang **trái** `k` vị trí, thêm `k` bit `0` bên phải ⇒ **nhân với `2^k`**.
- `x >> k` dịch sang **phải** `k` vị trí, bỏ `k` bit cuối ⇒ **chia nguyên cho `2^k`** (`x // 2**k`).
- `1 << p` chính là `2^p`: số có đúng một bit `1` ở vị trí `p`.

In [3]:
x = 5                      # 101
print(x << 1, bin(x << 1)) # 10  (1010)
print(x << 2, bin(x << 2)) # 20  (10100)
print(x >> 1, bin(x >> 1)) # 2   (10)
print(x >> 2, bin(x >> 2)) # 1   (1)

for p in range(0, 6):
    print(f"1 << {p} = {1 << p:>2}  (= 2^{p})")

10 0b1010
20 0b10100
2 0b10
1 0b1
1 << 0 =  1  (= 2^0)
1 << 1 =  2  (= 2^1)
1 << 2 =  4  (= 2^2)
1 << 3 =  8  (= 2^3)
1 << 4 = 16  (= 2^4)
1 << 5 = 32  (= 2^5)


### 3.3. Số toàn bit 1 chính là `2^p - 1`

Số có `p` bit, tất cả bằng `1`: `111...1` (p chữ số 1). Nếu cộng thêm `1` ta được `1000...0` (một bit `1` rồi `p` bit `0`), tức `2^p`. Vậy số toàn bit 1 là `2^p - 1`: `1, 3, 7, 15, 31, ...`.

In [4]:
for p in range(1, 11):
    v = (1 << p) - 1
    print(f"p={p:>2}: (1<<{p}) - 1 = {v:>4}  nhị phân = {bin(v)[2:]}")

p= 1: (1<<1) - 1 =    1  nhị phân = 1
p= 2: (1<<2) - 1 =    3  nhị phân = 11
p= 3: (1<<3) - 1 =    7  nhị phân = 111
p= 4: (1<<4) - 1 =   15  nhị phân = 1111
p= 5: (1<<5) - 1 =   31  nhị phân = 11111
p= 6: (1<<6) - 1 =   63  nhị phân = 111111
p= 7: (1<<7) - 1 =  127  nhị phân = 1111111
p= 8: (1<<8) - 1 =  255  nhị phân = 11111111
p= 9: (1<<9) - 1 =  511  nhị phân = 111111111
p=10: (1<<10) - 1 = 1023  nhị phân = 1111111111


### 3.4. `int.bit_length()`

`x.bit_length()` trả số bit cần để viết `x` (không tính các bit `0` ở đầu). Với `x >= 1` thì bit cao nhất luôn là `1`, và `x.bit_length() = len(bin(x)) - 2`. Ví dụ `5 = 101` có độ dài `3`; `10 = 1010` có độ dài `4`.

In [5]:
for x in (1, 2, 3, 4, 5, 7, 8, 10, 15, 16, 1000):
    print(f"x={x:>4}: bit_length={x.bit_length()}  len(bin)-2={len(bin(x)) - 2}  nhị phân={bin(x)[2:]}")

x=   1: bit_length=1  len(bin)-2=1  nhị phân=1
x=   2: bit_length=2  len(bin)-2=2  nhị phân=10
x=   3: bit_length=2  len(bin)-2=2  nhị phân=11
x=   4: bit_length=3  len(bin)-2=3  nhị phân=100
x=   5: bit_length=3  len(bin)-2=3  nhị phân=101
x=   7: bit_length=3  len(bin)-2=3  nhị phân=111
x=   8: bit_length=4  len(bin)-2=4  nhị phân=1000
x=  10: bit_length=4  len(bin)-2=4  nhị phân=1010
x=  15: bit_length=4  len(bin)-2=4  nhị phân=1111
x=  16: bit_length=5  len(bin)-2=5  nhị phân=10000
x=1000: bit_length=10  len(bin)-2=10  nhị phân=1111101000


### 3.5. Phép OR `|` và kiểm tra "toàn bit 1" bằng `x & (x + 1) == 0`

`a | b` đặt bit `1` ở mọi vị trí mà `a` hoặc `b` có bit `1`. Phép AND `&` giữ bit `1` ở vị trí **cả hai** có `1`.

Mẹo kiểm tra toàn bit 1: nếu `x = 2^p - 1` (toàn `1`) thì `x + 1 = 2^p` (một bit `1` rồi toàn `0`), hai số **không chung bit nào** ⇒ `x & (x + 1) = 0`. Nếu `x` có bit `0` ở đâu đó thì `x + 1` vẫn giữ lại ít nhất một bit `1` chung với `x`, nên `x & (x + 1) != 0`.

In [6]:
for x in (3, 5, 6, 7, 10, 15, 16):
    print(f"x={x:>2} {bin(x)[2:]:>5} & (x+1) {bin(x + 1)[2:]:>5} = {x & (x + 1):>2}  -> toàn bit 1? {x & (x + 1) == 0}")

x= 3    11 & (x+1)   100 =  0  -> toàn bit 1? True
x= 5   101 & (x+1)   110 =  4  -> toàn bit 1? False
x= 6   110 & (x+1)   111 =  6  -> toàn bit 1? False
x= 7   111 & (x+1)  1000 =  0  -> toàn bit 1? True
x=10  1010 & (x+1)  1011 = 10  -> toàn bit 1? False
x=15  1111 & (x+1) 10000 =  0  -> toàn bit 1? True
x=16 10000 & (x+1) 10001 = 16  -> toàn bit 1? False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `n = 5` (`101`): thử `5, 6, 7`.

| x | nhị phân | có bit 0? | toàn bit 1? |
|---|---|---|---|
| 5 | `101` | có | không |
| 6 | `110` | có | không |
| 7 | `111` | không | **có** — dừng |

Với `n = 10` (`1010`): thử `10, 11, 12, 13, 14, 15`; chỉ `15 = 1111` toàn bit 1.

Quan sát: các số toàn bit 1 thưa dần: `1, 3, 7, 15, 31, 63, 127, 255, 511, 1023`. Giữa `7` và `15` **không** có số nào thoả. Vì vậy ta không cần thử từng số — chỉ cần biết số toàn bit 1 **đầu tiên `>= n`** trong dãy trên. Với `n <= 1000` thì đáp án nhiều nhất là `1023`.

In [7]:
def show_candidates(n):
    x = n
    print(f"n = {n} ({bin(n)[2:]})")
    while True:
        ok = x & (x + 1) == 0
        print(f"  thử x={x:>4}  nhị phân={bin(x)[2:]:>11}  toàn bit 1? {ok}")
        if ok:
            return x
        x += 1

show_candidates(5)
show_candidates(10)
show_candidates(3)

n = 5 (101)
  thử x=   5  nhị phân=        101  toàn bit 1? False
  thử x=   6  nhị phân=        110  toàn bit 1? False
  thử x=   7  nhị phân=        111  toàn bit 1? True
n = 10 (1010)
  thử x=  10  nhị phân=       1010  toàn bit 1? False
  thử x=  11  nhị phân=       1011  toàn bit 1? False
  thử x=  12  nhị phân=       1100  toàn bit 1? False
  thử x=  13  nhị phân=       1101  toàn bit 1? False
  thử x=  14  nhị phân=       1110  toàn bit 1? False
  thử x=  15  nhị phân=       1111  toàn bit 1? True
n = 3 (11)
  thử x=   3  nhị phân=         11  toàn bit 1? True


3

## 5. Cách 1 — Brute Force / baseline: thử từng số từ `n` trở lên

**Ý tưởng.** Đặt `x = n`. Chừng nào `x` còn **ít nhất một bit `0`** thì `x += 1`. Khi `x` toàn bit 1 thì trả `x`. Kiểm tra "toàn bit 1" dùng `x & (x + 1) == 0` (mục 3.5) — hoặc dùng chuỗi `bin(x)[2:]` rồi kiểm tra không có ký tự `'0'`.

In [8]:
def smallest_bruteforce(n: int) -> int:
    x = n
    while x & (x + 1) != 0:     # còn bit 0 -> thử số kế tiếp
        x += 1
    return x

for n, expected in examples:
    print(n, "->", smallest_bruteforce(n), "(kỳ vọng", expected, ")")

5 -> 7 (kỳ vọng 7 )
10 -> 15 (kỳ vọng 15 )
3 -> 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Số lần lặp tối đa bằng khoảng cách từ `n` đến số toàn bit 1 kế tiếp, tức `O(n)` (ví dụ `n = 512` cần đi tới `1023`, `511` bước). Space `O(1)`. Với `n <= 1000` vẫn chạy tức thì.

**Nút thắt.** Ta đang đi qua **rất nhiều số vô ích**: từ `8` đến `14` không số nào thoả, nhưng ta vẫn kiểm tra từng số. Số toàn bit 1 là dãy **rất thưa**, và ta biết chính xác công thức của chúng (`2^p - 1`). Hãy nhảy thẳng tới đó.

## 6. Ý tưởng thuật toán / bước đột phá

**Chỉ cần tìm `p` nhỏ nhất sao cho `2^p - 1 >= n`.** Vì mọi số toàn bit 1 đều có dạng `2^p - 1`, đáp án là phần tử đầu tiên `>= n` của dãy tăng `1, 3, 7, 15, ...`.

Có hai cách nhìn đẹp:

1. **Dịch bit:** bắt đầu `x = 1`, mỗi lần `x <<= 1` (nhân đôi: `1, 2, 4, 8, ...`) cho đến khi `x - 1 >= n`. Khi đó `x - 1 = 2^p - 1`.
2. **Độ dài bit:** nếu `n` có `b` bit (`n.bit_length() = b`) thì `2^(b-1) <= n <= 2^b - 1`. Số toàn bit 1 trước đó là `2^(b-1) - 1 < n`, còn `2^b - 1 >= n`. Vậy đáp án đúng là `2^b - 1 = (1 << b) - 1`.

In [9]:
# Kiểm chứng ý tưởng 2: n có b bit  =>  2^(b-1) - 1 < n <= 2^b - 1
for n in (1, 2, 3, 4, 5, 7, 8, 10, 15, 16, 1000):
    b = n.bit_length()
    lo, hi = (1 << (b - 1)) - 1, (1 << b) - 1
    print(f"n={n:>4} b={b:>2}: {lo:>4} < n <= {hi:>4}  -> đáp án {hi}")
    assert lo < n <= hi

n=   1 b= 1:    0 < n <=    1  -> đáp án 1
n=   2 b= 2:    1 < n <=    3  -> đáp án 3
n=   3 b= 2:    1 < n <=    3  -> đáp án 3
n=   4 b= 3:    3 < n <=    7  -> đáp án 7
n=   5 b= 3:    3 < n <=    7  -> đáp án 7
n=   7 b= 3:    3 < n <=    7  -> đáp án 7
n=   8 b= 4:    7 < n <=   15  -> đáp án 15
n=  10 b= 4:    7 < n <=   15  -> đáp án 15
n=  15 b= 4:    7 < n <=   15  -> đáp án 15
n=  16 b= 5:   15 < n <=   31  -> đáp án 31
n=1000 b=10:  511 < n <= 1023  -> đáp án 1023


## 7. Các cách tiếp theo

### Cách 2 — Vòng lặp dịch bit (lời giải chính thức)

**Giải thích từng đoạn.**

1. `x = 1` — số `2^0`.
2. `while x - 1 < n:` — chừng nào `2^p - 1` còn **nhỏ hơn** `n`, chưa đủ lớn.
3. `x <<= 1` — nhân đôi `x`, tức tăng `p` thêm 1.
4. `return x - 1` — `x - 1 = 2^p - 1` là số toàn bit 1 đầu tiên `>= n`.

In [10]:
def smallest_shift(n: int) -> int:
    x = 1
    while x - 1 < n:
        x <<= 1
    return x - 1

for n, expected in examples:
    print(n, "->", smallest_shift(n), "(kỳ vọng", expected, ")")

5 -> 7 (kỳ vọng 7 )
10 -> 15 (kỳ vọng 15 )
3 -> 3 (kỳ vọng 3 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionShift:
    def smallestNumber(self, n: int) -> int:
        x = 1
        while x - 1 < n:
            x <<= 1
        return x - 1

print(SolutionShift().smallestNumber(10))   # 15

15


**Độ phức tạp Cách 2:** **Time `O(log n)`** (số lần nhân đôi bằng số bit của `n`), **Space `O(1)`**.

### Cách 3 — `bit_length`: một dòng

**Giải thích.** Dùng thẳng kết luận ở mục 6: `b = n.bit_length()` rồi trả `(1 << b) - 1`.

- `n = 5`: `b = 3`, `(1 << 3) - 1 = 7`.
- `n = 10`: `b = 4`, `(1 << 4) - 1 = 15`.
- `n = 3`: `b = 2`, `(1 << 2) - 1 = 3` (nó đã toàn bit 1 nên không đổi).

In [12]:
def smallest_bitlength(n: int) -> int:
    b = n.bit_length()
    return (1 << b) - 1

for n, expected in examples:
    print(n, "->", smallest_bitlength(n), "(kỳ vọng", expected, ")")

5 -> 7 (kỳ vọng 7 )
10 -> 15 (kỳ vọng 15 )
3 -> 3 (kỳ vọng 3 )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionBitLength:
    def smallestNumber(self, n: int) -> int:
        return (1 << n.bit_length()) - 1

print(SolutionBitLength().smallestNumber(5))   # 7

7


**Độ phức tạp Cách 3:** **Time `O(1)`** (với số nguyên cỡ máy), **Space `O(1)`**. Ngắn nhất, nhưng cần nhớ `bit_length` và công thức `2^b - 1`.

### Cách 4 — "Làm đầy" bit bằng OR và dịch phải

**Giải thích.** Bit cao nhất của `n` luôn là `1`. Nếu ta lấy `n | (n >> 1)` thì **hai bit cao nhất** đều là `1`; tiếp `| (n >> 2)` thì **bốn bit cao nhất**; `| (n >> 4)` thì **tám bit cao nhất**... Sau vài lần ta có `1` ở **mọi** vị trí từ bit cao nhất xuống bit `0`, tức đúng `2^b - 1`.

Ví dụ `n = 10 = 1010`: `1010 | 0101 = 1111`.
Ví dụ `n = 5 = 101`: `101 | 010 = 111`.

Vì `n <= 1000` (10 bit), các lần dịch `1, 2, 4, 8` là đủ (ta dùng thêm `16` cho an toàn với số 32 bit).

In [14]:
def smallest_smear(n: int) -> int:
    x = n
    for sh in (1, 2, 4, 8, 16):
        x |= x >> sh
    return x

for n, expected in examples:
    print(n, "->", smallest_smear(n), "(kỳ vọng", expected, ")")

# Quan sát từng bước với n = 10
x = 10
print(f"bắt đầu: {x:>4} = {bin(x)[2:]:>6}")
for sh in (1, 2, 4):
    x |= x >> sh
    print(f"x |= x >> {sh}: {x:>3} = {bin(x)[2:]:>6}")

5 -> 7 (kỳ vọng 7 )
10 -> 15 (kỳ vọng 15 )
3 -> 3 (kỳ vọng 3 )
bắt đầu:   10 =   1010
x |= x >> 1:  15 =   1111
x |= x >> 2:  15 =   1111
x |= x >> 4:  15 =   1111


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionSmear:
    def smallestNumber(self, n: int) -> int:
        x = n
        for sh in (1, 2, 4, 8, 16):
            x |= x >> sh
        return x

print(SolutionSmear().smallestNumber(10))   # 15

15


**Độ phức tạp Cách 4:** **Time `O(log w)`** với `w` là số bit của kiểu số (ở đây hằng số, 5 bước), **Space `O(1)`**. Đây là mẹo kinh điển trong các bài bit ("làm đầy" bit).

### Cách 5 — Tra bảng 10 số toàn bit 1

**Giải thích.** Vì `n <= 1000`, chỉ có 10 số toàn bit 1 liên quan: `1, 3, 7, ..., 1023`. Ta tạo danh sách này rồi lấy **phần tử đầu tiên `>= n`** (hoặc dùng `bisect_left`).

In [16]:
from bisect import bisect_left

TABLE = [(1 << p) - 1 for p in range(1, 11)]
print(TABLE)       # [1, 3, 7, 15, 31, 63, 127, 255, 511, 1023]

def smallest_table(n: int) -> int:
    return TABLE[bisect_left(TABLE, n)]

for n, expected in examples:
    print(n, "->", smallest_table(n), "(kỳ vọng", expected, ")")

[1, 3, 7, 15, 31, 63, 127, 255, 511, 1023]
5 -> 7 (kỳ vọng 7 )
10 -> 15 (kỳ vọng 15 )
3 -> 3 (kỳ vọng 3 )


In [17]:
# Code nộp LeetCode cho Cách 5
from bisect import bisect_left

class SolutionTable:
    def smallestNumber(self, n: int) -> int:
        table = [(1 << p) - 1 for p in range(1, 11)]   # 1, 3, 7, ..., 1023
        return table[bisect_left(table, n)]

print(SolutionTable().smallestNumber(1000))   # 1023

1023


**Độ phức tạp Cách 5:** **Time `O(log 10)` ≈ `O(1)`**, **Space `O(1)`** (bảng cố định). `bisect_left(a, n)` trả vị trí đầu tiên mà `a[i] >= n`. Cách này chỉ hợp lý nhờ ràng buộc nhỏ; nếu `n` lớn, hãy dùng Cách 2 hoặc 3.

## 8. TRACE từng bước

Trace **Cách 2** với `n = 10`:

| bước | x | x - 1 | `x - 1 < n`? | hành động |
|---|---|---|---|---|
| 0 | 1 | 0 | `0 < 10` có | `x <<= 1` → 2 |
| 1 | 2 | 1 | `1 < 10` có | `x <<= 1` → 4 |
| 2 | 4 | 3 | `3 < 10` có | `x <<= 1` → 8 |
| 3 | 8 | 7 | `7 < 10` có | `x <<= 1` → 16 |
| 4 | 16 | 15 | `15 < 10` **không** | dừng, trả `16 - 1 = 15` |

Còn với `n = 3`: `x=1 (0<3)` → `2 (1<3)` → `4 (3<3 sai)` → trả `3`. Điều kiện **`<` nghiêm ngặt** là lý do `n = 3` giữ nguyên (nếu dùng `<=` ta đi quá và trả `7`).

In [18]:
def trace_shift(n):
    x = 1
    step = 0
    print(f"n = {n} ({bin(n)[2:]})")
    while True:
        cond = x - 1 < n
        print(f"bước {step}: x={x:>5} ({bin(x)[2:]:>6})  x-1={x - 1:>4}  x-1 < n ? {cond}")
        if not cond:
            break
        x <<= 1
        step += 1
    print(f"-> trả x - 1 = {x - 1} ({bin(x - 1)[2:]})")
    return x - 1

trace_shift(10)
trace_shift(3)
trace_shift(1000)

n = 10 (1010)
bước 0: x=    1 (     1)  x-1=   0  x-1 < n ? True
bước 1: x=    2 (    10)  x-1=   1  x-1 < n ? True
bước 2: x=    4 (   100)  x-1=   3  x-1 < n ? True
bước 3: x=    8 (  1000)  x-1=   7  x-1 < n ? True
bước 4: x=   16 ( 10000)  x-1=  15  x-1 < n ? False
-> trả x - 1 = 15 (1111)
n = 3 (11)
bước 0: x=    1 (     1)  x-1=   0  x-1 < n ? True
bước 1: x=    2 (    10)  x-1=   1  x-1 < n ? True
bước 2: x=    4 (   100)  x-1=   3  x-1 < n ? False
-> trả x - 1 = 3 (11)
n = 1000 (1111101000)
bước 0: x=    1 (     1)  x-1=   0  x-1 < n ? True
bước 1: x=    2 (    10)  x-1=   1  x-1 < n ? True
bước 2: x=    4 (   100)  x-1=   3  x-1 < n ? True
bước 3: x=    8 (  1000)  x-1=   7  x-1 < n ? True
bước 4: x=   16 ( 10000)  x-1=  15  x-1 < n ? True
bước 5: x=   32 (100000)  x-1=  31  x-1 < n ? True
bước 6: x=   64 (1000000)  x-1=  63  x-1 < n ? True
bước 7: x=  128 (10000000)  x-1= 127  x-1 < n ? True
bước 8: x=  256 (100000000)  x-1= 255  x-1 < n ? True
bước 9: x=  512 (1000000000)  x

1023

In [19]:
# Trace Cách 4 (làm đầy bit) với n = 1000 = 1111101000
def trace_smear(n):
    x = n
    print(f"bắt đầu: x={x:>5} = {bin(x)[2:]:>10}")
    for sh in (1, 2, 4, 8):
        x |= x >> sh
        print(f"x |= x >> {sh}: x={x:>5} = {bin(x)[2:]:>10}")
    return x

assert trace_smear(1000) == 1023

bắt đầu: x= 1000 = 1111101000
x |= x >> 1: x= 1020 = 1111111100
x |= x >> 2: x= 1023 = 1111111111
x |= x >> 4: x= 1023 = 1111111111
x |= x >> 8: x= 1023 = 1111111111


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (vòng lặp dịch bit): rõ nghĩa, chứng minh dễ, không cần nhớ hàm đặc biệt. Đây là lời giải chính thức; Cách 3 là biến thể một dòng.

In [20]:
class Solution:
    def smallestNumber(self, n: int) -> int:
        x = 1                      # x = 2^p
        while x - 1 < n:           # 2^p - 1 còn nhỏ hơn n
            x <<= 1                # p tăng 1  (x nhân đôi)
        return x - 1               # 2^p - 1: toàn bit 1, nhỏ nhất mà >= n


for n, expected in examples:
    got = Solution().smallestNumber(n)
    print(n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

5 -> 7 OK
10 -> 15 OK
3 -> 3 OK


## 10. Vì sao thuật toán đúng

**Bước 1 — Dạng của số toàn bit 1.** Số có `p >= 1` bit, tất cả bằng `1`, có giá trị `2^(p-1) + ... + 2 + 1 = 2^p - 1`. Ngược lại mọi số `2^p - 1` đều toàn bit 1. Vậy tập đáp án khả dĩ chính là `{1, 3, 7, 15, ...}`.

**Bước 2 — Dãy tăng ngặt.** Dãy `2^p - 1` tăng ngặt theo `p`, nên "số nhỏ nhất `>= n`" chính là phần tử **đầu tiên** của dãy thoả `2^p - 1 >= n`.

**Bước 3 — Vòng lặp tìm đúng phần tử đầu tiên.** Bất biến: trước mỗi lần kiểm tra, `x = 2^p` và mọi `2^q - 1` với `q < p` đều `< n` (vì ta chưa dừng). Khi `x - 1 >= n` lần đầu, `2^p - 1` là phần tử đầu tiên `>= n` ⇒ đúng đáp án. Vòng lặp luôn dừng vì `x` tăng gấp đôi mỗi lần.

**Biên.** `n = 1`: `x = 1`, `x - 1 = 0 < 1` ⇒ `x = 2`, `x - 1 = 1 < 1` sai ⇒ trả `1`. Đúng.

## 11. Phân tích Time / Space Complexity

Gọi `b = n.bit_length() ≈ log2(n)`.

- **Cách 1 (thử từng số):** Time `O(n)`, Space `O(1)`.
- **Cách 2 (dịch bit):** Time `O(log n)`, Space `O(1)`.
- **Cách 3 (`bit_length`):** Time `O(1)`, Space `O(1)`.
- **Cách 4 (làm đầy bit):** Time `O(log w)` (hằng số), Space `O(1)`.
- **Cách 5 (tra bảng + bisect):** Time `O(1)` theo ràng buộc, Space `O(1)`.

Với `n <= 1000` thì `b <= 10`, vòng lặp Cách 2 chạy tối đa 10 lần. Mọi cách đều nhanh; điểm khác nhau chỉ là độ **tổng quát** (Cách 2, 3, 4 dùng được với `n` rất lớn nhờ số nguyên Python không giới hạn).

## 12. Bộ test

### 12.1. Ví dụ của đề và các trường hợp biên

Ta chạy mọi cách trên các test; đáp án đã được tính tay trong notebook (theo dãy `1, 3, 7, ..., 1023`).

In [21]:
impls = {
    "brute":   smallest_bruteforce,
    "shift":   lambda n: Solution().smallestNumber(n),
    "bitlen":  lambda n: SolutionBitLength().smallestNumber(n),
    "smear":   lambda n: SolutionSmear().smallestNumber(n),
    "table":   lambda n: SolutionTable().smallestNumber(n),
}

cases = [
    (5, 7), (10, 15), (3, 3),          # ví dụ đề
    (1, 1),                            # tự dựng: nhỏ nhất
    (2, 3),                            # tự dựng: 10 -> 11
    (4, 7),                            # tự dựng: lũy thừa của 2 -> lên 2^(b) - 1
    (7, 7), (8, 15),                   # tự dựng: biên giữa 7 và 8
    (15, 15), (16, 31),                # tự dựng
    (511, 511), (512, 1023),           # tự dựng: sát 1023
    (1000, 1023),                      # tự dựng: ràng buộc lớn nhất
    (1023 - 1, 1023),                  # tự dựng: 1022 -> 1023 (nằm ngoài n <= 1000 nhưng vẫn đúng)
]

for n, exp in cases:
    for name, f in impls.items():
        got = f(n)
        assert got == exp, (name, n, got, exp)
print("12.1 OK:", len(cases), "test x", len(impls), "cách")

12.1 OK: 14 test x 5 cách


### 12.2. Phản ví dụ cho các cách làm sai

Ta viết các bản sai có chủ đích rồi tìm input làm lộ lỗi, để chắc chắn bộ test đủ mạnh.

In [22]:
def wrong_strict(n):
    """Sai: dùng <= thay <  (lệch một đơn vị)."""
    x = 1
    while x - 1 <= n:
        x <<= 1
    return x - 1

def wrong_bin_prefix(n):
    """Sai: kiểm tra '0' not in bin(x) -> '0b' luôn có ký tự '0'."""
    x = n
    while "0" in bin(x):
        x += 1
        if x > 5000:          # chặn vòng lặp vô hạn để minh hoạ
            return None
    return x

def wrong_no_minus(n):
    """Sai: trả 2^b thay vì 2^b - 1."""
    return 1 << n.bit_length()

def wrong_precedence(n):
    """Sai: 1 << n.bit_length() - 1 được hiểu là 1 << (bit_length - 1)."""
    return 1 << n.bit_length() - 1

print("strict    :", wrong_strict(3), "(đúng phải 3)")
print("bin_prefix:", wrong_bin_prefix(7), "(đúng phải 7)")
print("no_minus  :", wrong_no_minus(5), "(đúng phải 7)")
print("precedence:", wrong_precedence(5), "(đúng phải 7)")
assert wrong_strict(3) == 7 and smallest_bruteforce(3) == 3
assert wrong_bin_prefix(7) is None
assert wrong_no_minus(5) == 8
assert wrong_precedence(5) == 4
print("12.2 OK: bốn bản sai đều bị lộ")

strict    : 7 (đúng phải 3)
bin_prefix: None (đúng phải 7)
no_minus  : 8 (đúng phải 7)
precedence: 4 (đúng phải 7)
12.2 OK: bốn bản sai đều bị lộ


### 12.3. Vét cạn toàn bộ `n = 1..1000` và đối chiếu ngẫu nhiên (seed)

Vì miền `n` chỉ có 1000 giá trị, ta kiểm tra **tất cả** bằng một **hàm kiểm tra hợp lệ độc lập**: `x` hợp lệ khi (a) `x >= n`, (b) mọi bit của `x` là `1` (kiểm bằng chuỗi nhị phân), (c) không có số toàn bit 1 nào nằm trong `[n, x - 1]` (tức `x` nhỏ nhất). Sau đó đối chiếu ngẫu nhiên có seed.

In [23]:
import random

def is_valid_answer(n, x):
    s = format(x, "b")
    if x < n or "0" in s:
        return False
    # nhỏ nhất: không có số toàn bit 1 trong [n, x-1]
    for y in range(n, x):
        if "0" not in format(y, "b"):
            return False
    return True

for n in range(1, 1001):
    answers = {name: f(n) for name, f in impls.items()}
    assert len(set(answers.values())) == 1, (n, answers)
    assert is_valid_answer(n, answers["shift"]), (n, answers)
print("12.3a OK: n = 1..1000, 5 cách trùng nhau và hợp lệ")

random.seed(3370)
for _ in range(500):
    n = random.randint(1, 1000)
    assert impls["shift"](n) == impls["bitlen"](n) == impls["smear"](n) == impls["table"](n) == impls["brute"](n)
print("12.3b OK: 500 input ngẫu nhiên (seed 3370)")

# Với n lớn (ngoài ràng buộc) 3 cách tổng quát vẫn khớp nhau
for n in (10**6, 2**20, 2**20 + 1, 2**30 - 1, 10**9):
    a, b, c = impls["shift"](n), impls["bitlen"](n), impls["smear"](n)
    assert a == b == c and "0" not in format(a, "b") and a >= n
print("12.3c OK: n lớn, cách 2/3/4 khớp nhau")

12.3a OK: n = 1..1000, 5 cách trùng nhau và hợp lệ
12.3b OK: 500 input ngẫu nhiên (seed 3370)
12.3c OK: n lớn, cách 2/3/4 khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra `'0' not in bin(x)` mà quên tiền tố `'0b'`

`bin(7)` là `'0b111'` — chuỗi này **luôn có ký tự `'0'`** (trong `'0b'`). Do đó điều kiện "không có số 0" **không bao giờ đúng**, vòng lặp chạy mãi.

In [24]:
print(bin(7), "0" in bin(7))           # '0b111' True  -> dù 7 toàn bit 1!

# SAI: vòng lặp không bao giờ dừng (ta chặn bằng bộ đếm để minh hoạ)
x, steps = 7, 0
while "0" in bin(x) and steps < 5:     # '0' in '0b...' luôn True
    x += 1
    steps += 1
print("đi lệch tới x =", x, "sau", steps, "bước (đáng lẽ dừng ngay ở 7)")

# ĐÚNG: bỏ tiền tố trước khi kiểm tra
print("0" in bin(7)[2:])               # False -> toàn bit 1
print("0" in format(7, "b"))           # False

0b111 True
đi lệch tới x = 12 sau 5 bước (đáng lẽ dừng ngay ở 7)
False
False


### Lỗi 2 — Dùng `<=` thay `<` trong vòng lặp (lệch một đơn vị)

Nếu `x - 1` **đã bằng `n`** thì ta đã tìm thấy rồi, không cần nhân đôi nữa. Dùng `<=` làm `n = 3` bị trả `7`.

In [25]:
# SAI
def sai_2(n):
    x = 1
    while x - 1 <= n:        # <-- sai: nên là <
        x <<= 1
    return x - 1

print(sai_2(3), sai_2(7), sai_2(1))    # 7 15 3  (sai: đúng là 3 7 1)

# ĐÚNG
print(Solution().smallestNumber(3), Solution().smallestNumber(7), Solution().smallestNumber(1))   # 3 7 1

7 15 3
3 7 1


### Lỗi 3 — Sai thứ tự ưu tiên: `1 << n.bit_length() - 1`

Trong Python, `-` ưu tiên **cao hơn** `<<`. Vậy `1 << b - 1` nghĩa là `1 << (b - 1)`, **không** phải `(1 << b) - 1`. Luôn thêm ngoặc.

In [26]:
b = 3
print(1 << b - 1)       # 4  (tức 1 << (b-1))  -> sai
print((1 << b) - 1)     # 7  -> đúng

# SAI
def sai_3(n):
    return 1 << n.bit_length() - 1

print(sai_3(5), "(sai; đúng là 7)")
print(Solution().smallestNumber(5))

4
7
4 (sai; đúng là 7)
7


### Lỗi 4 — Nhầm `2^p` và `2^p - 1`

`1 << b` là `1000...0` (có `b` bit `0` sau số `1`), **không phải** toàn bit 1. Đừng quên trừ đi `1`.

In [27]:
for n in (5, 10, 3):
    b = n.bit_length()
    print(f"n={n}: 1<<b = {1 << b:>2} ({bin(1 << b)[2:]})  còn  (1<<b)-1 = {(1 << b) - 1:>2} ({bin((1 << b) - 1)[2:]})")

n=5: 1<<b =  8 (1000)  còn  (1<<b)-1 =  7 (111)
n=10: 1<<b = 16 (10000)  còn  (1<<b)-1 = 15 (1111)
n=3: 1<<b =  4 (100)  còn  (1<<b)-1 =  3 (11)


### Lỗi 5 — Giả sử đáp án luôn **lớn hơn** `n`

Đề dùng "**greater than or equal**": nếu `n` đã toàn bit 1 (như `n = 3, 7, 15`) thì đáp án chính là `n`. Bắt đầu thử từ `n + 1` sẽ sai ở những input này.

In [28]:
def sai_5(n):
    x = n + 1                 # <-- sai: bỏ qua trường hợp x = n
    while x & (x + 1) != 0:
        x += 1
    return x

print(sai_5(3), sai_5(7), "(sai; đúng là 3 và 7)")
print(Solution().smallestNumber(3), Solution().smallestNumber(7))

7 15 (sai; đúng là 3 và 7)
3 7


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "**toàn bit 1**", "**mask toàn 1**" ⇒ nghĩ ngay `2^p - 1` (số "Mersenne"), viết là `(1 << p) - 1`.
- Cần **số bit** của `n` ⇒ `n.bit_length()` thay vì tự đếm bằng vòng lặp.
- Muốn biết "`x` là toàn bit 1?" ⇒ `x & (x + 1) == 0` (và `x > 0`). Tương tự, "`x` là luỹ thừa của 2?" ⇒ `x & (x - 1) == 0`.
- "Làm đầy" bit sau bit cao nhất bằng `x |= x >> 1; x |= x >> 2; ...`.
- Bài bit có ràng buộc nhỏ (`n <= 1000`) cho phép cả vét cạn lẫn tra bảng; hãy dùng vét cạn để **kiểm chứng** công thức.
- Khi gặp bài bit, hãy **in nhị phân** (`bin`, `format`) để nhìn thấy cấu trúc; đa số bài sẽ "lộ" quy luật ngay khi đó.
- Chú ý thứ tự ưu tiên toán tử bit trong Python: `+ -` ưu tiên cao hơn `<< >>`, rồi tới `&`, `^`, `|`; luôn dùng ngoặc khi nghi ngờ.

## 15. Biến thể của bài

- **Số toàn bit 1 lớn nhất `<= n`:** `(1 << n.bit_length()) - 1` nếu `n` toàn bit 1, ngược lại `(1 << (n.bit_length() - 1)) - 1` (bài tập 4).
- **Phần bù nhị phân (Complement of Base 10 Integer, LeetCode 1009):** `n ^ smallestNumber(n)` — lật mọi bit của `n` (bài tập 5).
- **Số nhỏ nhất `>= n` mà nhị phân là luỹ thừa của 2:** `1 << (n - 1).bit_length()`.
- **Đếm số `x` trong `[1, n]` toàn bit 1:** `n.bit_length()` hoặc `n.bit_length() - 1` tuỳ `n` có toàn bit 1 hay không.
- **Số nhỏ nhất `>= n` có đúng `k` bit 1:** bài khó hơn, phải thao tác bit từng vị trí.
- **Mở rộng cho `n` rất lớn (`10^18`):** Cách 2, 3, 4 vẫn dùng được vì số nguyên Python không tràn.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng số | Tăng `x` từ `n` cho đến khi `x & (x+1) == 0` | `O(n)` | `O(1)` | Để kiểm chứng; `n` nhỏ |
| 2. Dịch bit | Nhân đôi `x` đến khi `x - 1 >= n` | `O(log n)` | `O(1)` | **Cách nộp chính**: dễ chứng minh |
| 3. `bit_length` | `(1 << n.bit_length()) - 1` | `O(1)` | `O(1)` | Khi muốn ngắn gọn, đã quen bit |
| 4. Làm đầy bit | `x |= x >> 1, 2, 4, ...` | `O(log w)` | `O(1)` | Mẹo bit kinh điển; hữu ích ở bài khác |
| 5. Tra bảng | 10 số `2^p - 1` + `bisect` | `O(1)` | `O(1)` | Khi ràng buộc nhỏ cố định |

Tất cả cùng đáp án (đã vét cạn `n = 1..1000` ở mục 12.3).

## 17. 5 bài tập vận dụng (dễ → khó)

Các cell TODO **không được gọi khi chạy notebook** — hãy tự điền rồi tự chạy. Đáp án ở mục 19.

### Bài 1 — Đoán output

Đoán kết quả `bai1()` (không chạy) — gợi ý: viết mỗi `n` dưới dạng nhị phân rồi đếm số bit.

In [29]:
def bai1():
    out = []
    for n in (1, 6, 8, 100):
        out.append((1 << n.bit_length()) - 1)
    return out

# Hãy đoán rồi mới chạy:  print(bai1())

### Bài 2 — Tính tay

Với `n = 100`: viết `n` ở dạng nhị phân, đếm số bit, rồi tính đáp án của bài (số toàn bit 1 nhỏ nhất `>= 100`), cuối cùng viết đáp án ở nhị phân. Điền vào ba biến dưới đây (không chạy hàm để lấy đáp án).

In [30]:
# TODO: tự tính tay rồi điền
bai2_bin = None       # chuỗi nhị phân của 100
bai2_ans = None       # đáp án (số nguyên)
bai2_ans_bin = None   # chuỗi nhị phân của đáp án

### Bài 3 — Điền TODO

Viết lại lời giải bằng vòng lặp `while` nhưng **không dùng `<<`**: dùng nhân `* 2` để tạo `2^p`. Hàm `bai3(n)` trả đáp án của bài (`2^p - 1` nhỏ nhất `>= n`).

In [31]:
def bai3(n):
    p2 = 1                      # p2 = 2^p
    # TODO: lặp trong khi p2 - 1 < n:  p2 *= 2
    ...
    # TODO: trả p2 - 1
    ...

### Bài 4 — Tự viết lại

Viết `bai4(n)` trả về số toàn bit 1 **lớn nhất không vượt quá `n`** (với `n >= 1`). Ví dụ `bai4(5) = 3`, `bai4(10) = 7`, `bai4(7) = 7`, `bai4(1) = 1`.

In [32]:
def bai4(n):
    # TODO: tự viết
    ...

### Bài 5 — Biến thể / mini challenge

Viết `bai5(n)` trả về **phần bù nhị phân** của `n`: lật mọi bit trong phạm vi độ dài nhị phân của `n`. Ví dụ `bai5(5) = 2` (`101 -> 010`), `bai5(10) = 5` (`1010 -> 0101`), `bai5(7) = 0`. Dùng ý tưởng của bài này (gợi ý: XOR với số toàn bit 1 cùng độ dài).

In [33]:
def bai5(n):
    # TODO: tự viết
    ...

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.**
- Mức 1: viết `1, 6, 8, 100` ở dạng nhị phân: `1`, `110`, `1000`, `1100100`.
- Mức 2: số bit lần lượt là `1, 3, 4, 7`.
- Mức 3: đáp án `(1 << b) - 1` cho từng `b`: `[1, 7, 15, 127]`.

**Bài 2.**
- Mức 1: `100 = 64 + 32 + 4 = 1100100` (7 bit).
- Mức 2: toàn bit 1 nhỏ hơn `100` là `63 (6 bit)`; số 7 bit toàn bit 1 là `127`.
- Mức 3: đáp án `127 = 1111111`.

**Bài 3.**
- Mức 1: giống Cách 2 nhưng thay `x <<= 1` bằng `p2 *= 2`.
- Mức 2: điều kiện lặp `p2 - 1 < n` (nghiêm ngặt).
- Mức 3: `while p2 - 1 < n: p2 *= 2` rồi `return p2 - 1`.

**Bài 4.**
- Mức 1: nếu `n` toàn bit 1 thì đáp án là `n` (kiểm bằng `n & (n + 1) == 0`).
- Mức 2: nếu không, số toàn bit 1 lớn nhất `<= n` có **ít hơn** một bit so với `n`.
- Mức 3: `b = n.bit_length()`; nếu `n & (n + 1) == 0` trả `n`, ngược lại trả `(1 << (b - 1)) - 1`.

**Bài 5.**
- Mức 1: lật bit nghĩa là XOR với `1` ở các vị trí cần lật.
- Mức 2: mask `m` = số toàn bit 1 cùng độ dài với `n`: `(1 << n.bit_length()) - 1`.
- Mức 3: trả `n ^ m`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án đều được kiểm tra bằng `assert`.

In [34]:
# --- Bài 1 ---
assert bai1() == [1, 7, 15, 127]
print("Bài 1:", bai1())

# --- Bài 2 ---
bai2_bin_da = "1100100"
bai2_ans_da = 127
bai2_ans_bin_da = "1111111"
assert format(100, "b") == bai2_bin_da
assert Solution().smallestNumber(100) == bai2_ans_da
assert format(bai2_ans_da, "b") == bai2_ans_bin_da
print("Bài 2:", bai2_bin_da, bai2_ans_da, bai2_ans_bin_da)

# --- Bài 3 ---
def bai3_da(n):
    p2 = 1
    while p2 - 1 < n:
        p2 *= 2
    return p2 - 1

for n in range(1, 1001):
    assert bai3_da(n) == Solution().smallestNumber(n)
print("Bài 3 OK (n = 1..1000)")

# --- Bài 4 ---
def bai4_da(n):
    if n & (n + 1) == 0:
        return n
    return (1 << (n.bit_length() - 1)) - 1

def oracle4(n):
    return max(x for x in range(1, n + 1) if "0" not in format(x, "b"))

for n in range(1, 1001):
    assert bai4_da(n) == oracle4(n), n
assert (bai4_da(5), bai4_da(10), bai4_da(7), bai4_da(1)) == (3, 7, 7, 1)
print("Bài 4 OK")

# --- Bài 5 ---
def bai5_da(n):
    return n ^ ((1 << n.bit_length()) - 1)

assert (bai5_da(5), bai5_da(10), bai5_da(7)) == (2, 5, 0)
for n in range(1, 1001):
    flipped = "".join("1" if c == "0" else "0" for c in format(n, "b"))
    assert bai5_da(n) == int(flipped, 2), n
print("Bài 5 OK")

Bài 1: [1, 7, 15, 127]
Bài 2: 1100100 127 1111111
Bài 3 OK (n = 1..1000)


Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (dịch bit):**

```python
class Solution:
    def smallestNumber(self, n: int) -> int:
        x = 1
        while x - 1 < n:
            x <<= 1
        return x - 1
```

**Biến thể một dòng:** `return (1 << n.bit_length()) - 1`.

**Checklist trước khi submit:**

- [ ] Tên method đúng: `smallestNumber(self, n)`.
- [ ] Điều kiện vòng lặp là `x - 1 < n` (nghiêm ngặt, không phải `<=`).
- [ ] Kết quả là `x - 1`, không phải `x`.
- [ ] Nếu dùng `bit_length`: có ngoặc `(1 << b) - 1`.
- [ ] Không kiểm tra `'0' in bin(x)` (nhớ `'0b'`).
- [ ] Đã thử: `n = 1`, `n = 3` (đã toàn bit 1), `n = 4`, `n = 512`, `n = 1000`.